In [2]:
import sys, os, time, math, csv
import pandas as pd
import numpy as np
from os.path import join
from collections import Counter

sys.path.append("/mnt/hbnas/home/fgan/Proj-Entropy/Video_Classification_Model/src")


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score

import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('agg')
plt.rcParams["font.family"] = "sans-serif"

from model_resnet_tsc import ResNet


In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
root = "/mnt/hbnas/home/fgan/Proj-Entropy/Video_Classification_Model"

In [8]:
def calculate_metrics(y_true, y_pred, duration, y_true_val=None, y_pred_val=None):
    res = pd.DataFrame(data=np.zeros((1, 4), dtype=np.float32), index=[0],
                       columns=['precision', 'accuracy', 'recall', 'duration'])
    res['precision'] = precision_score(y_true, y_pred, average='macro')
    res['accuracy'] = accuracy_score(y_true, y_pred)

    if not y_true_val is None:
        # this is useful when transfer learning is used with cross validation
        res['accuracy_val'] = accuracy_score(y_true_val, y_pred_val)

    res['recall'] = recall_score(y_true, y_pred, average='macro')
    res['duration'] = duration
    
    # quick formatting for latex
    res['precision'] = round(res['precision'], 4) * 100
    res['accuracy'] = round(res['accuracy'], 4) * 100
    res['recall'] = round(res['recall'], 4) * 100
    
    return res

In [12]:
# first, check out couple of models and load pre-trained weights

cases = ["hevc_marginally_imbalanced_800K", 
         "hevc_marginally_imbalanced_1000K", 
         "hevc_marginally_imbalanced_1200K", 
         "hevc_all", 
         "h264_all"]

model_test = ResNet(input_shape=(1,3000), num_classes=11, n_feature_maps=64).to(device)

for case in cases:
    checkpoint = torch.load(join(root, f'4_checkpoints/resnet/penguin_archive/{case}/best_model.pt'))
    model_test.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    criterion = nn.CrossEntropyLoss()

    print(">", case, "<")
    for k, v in checkpoint.items():
        if "state_dict" in k:
            continue
        print(k, v)
    print()


> hevc_marginally_imbalanced_800K <
epoch 186
train_loss tensor(4.3549e-05, device='cuda:0', requires_grad=True)
val_loss tensor(0.7659, device='cuda:0')
train_acc 0.9998568565702834
val_acc 0.8427835051546392

> hevc_marginally_imbalanced_1000K <
epoch 243
train_loss tensor(1.3078e-05, device='cuda:0', requires_grad=True)
val_loss tensor(0.7229, device='cuda:0')
train_acc 1.0
val_acc 0.8466494845360825

> hevc_marginally_imbalanced_1200K <
epoch 101
train_loss tensor(9.8143e-05, device='cuda:0', requires_grad=True)
val_loss tensor(0.7101, device='cuda:0')
train_acc 0.9991411394217006
val_acc 0.8350515463917526

> hevc_all <
epoch 167
train_loss tensor(3.6598e-06, device='cuda:0', requires_grad=True)
val_loss tensor(0.6325, device='cuda:0')
train_acc 1.0
val_acc 0.8904739990796134

> h264_all <
epoch 103
train_loss tensor(9.4476e-06, device='cuda:0', requires_grad=True)
val_loss tensor(0.4895, device='cuda:0')
train_acc 1.0
val_acc 0.9002298850574713



In [6]:
def run_test(X_path, y_path, model_test, criterion):
    # Load data from the DataFrame
    X_test = pd.read_csv(X_path).values
    y_test = pd.read_csv(y_path).values

    # normalize X (each example individually), 
    # and expand X to include a C dimension, models expects it
    X_test_mean, X_test_std = X_test.mean(axis=1, keepdims=True), X_test.std(axis=1, keepdims=True)
    X_test = (X_test - X_test_mean) / X_test_std
    X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

    # and one-hot encode y, to calculate cross-entropy loss
    enc = OneHotEncoder(categories='auto')
    y_test = enc.fit_transform(y_test.reshape(-1, 1)).toarray()
    nb_classes = len(enc.categories_[0])

    # a quick stats check of our test set
    # print("samples:", len(X_test), ", num_classes", nb_classes)
    # print("labels for couple random samples: ", 
    #     enc.categories_[0][y_test.argmax(axis=1)][np.random.randint(0, len(y_test), 20)])
    
    
    ## creating data loaders
    # Convert the numpy array to a PyTorch tensor
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                torch.tensor(y_test, dtype=torch.float32))

    ### Create a DataLoader from the TensorDataset
    batch_size = 1024
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    
    # and run the test dataset through our saved model
    model_test.eval()
    with torch.no_grad():
        test_loss = 0     # for lr scheduler
        test_logits, test_truth = [], []
        for batch_num, (X, y) in enumerate(test_dataloader):
            print(f"{batch_num}/{len(test_dataloader)}", end="\r")
            X, y = X.to(device), y.to(device)
            logits = model_test(X)
            test_logits += logits
            test_truth += y
            test_loss += criterion(logits, y) * len(y)     # properly weighting each batch loss, 
        test_loss /= len(test_dataloader.dataset)

    # print(test_loss)

    pred = torch.stack(test_logits).max(dim=1)[1].cpu().numpy()
    truth = torch.stack(test_truth).max(dim=1)[1].cpu().numpy()

    print(calculate_metrics(pred, truth, 0.0))
    
    return
    # now calculate accuracies for each class
    total = Counter(truth)      # total number of samples for each category, taken from Ground Truth
                                # so... RECALL?
    correct_pred = Counter(truth[truth == pred])

    stats = [ {"category": enc.categories_[0][numeric_cat],
            "correct": correct_pred[numeric_cat], 
            "total_samples": total[numeric_cat], 
            "accuracy": correct_pred[numeric_cat]/total[numeric_cat]}
                for numeric_cat in total]

    for x in sorted(stats, key=lambda x: x["accuracy"], reverse=True):
        print(f"Accuracy for {x['category']:<15}= {x['accuracy']:.2%}  <--  {x['correct']}/{x['total_samples']}")




In [9]:
# the heavy lifter, any combination of case=model and data=test set

cases = [
    "hevc_marginally_imbalanced_800K_withB", 
    "hevc_marginally_imbalanced_1000K_withB", 
    "hevc_marginally_imbalanced_1200K_withB",
    "hevc_marginally_imbalanced_1500K_withB",
    # "hevc_all", 
    # "h264_all"
]

data = [
    "hevc_marginally_imbalanced_800K_withB", 
    "hevc_marginally_imbalanced_1000K_withB", 
    "hevc_marginally_imbalanced_1200K_withB",
    "hevc_marginally_imbalanced_1500K_withB",
    # "hevc_all", 
    # "h264_all"
]

for case in cases:
    try:
        model_test = ResNet(input_shape=(1,3000), num_classes=11, n_feature_maps=64).to(device)
        checkpoint = torch.load(join(root, f'4_checkpoints/resnet/penguin_archive/{case}/best_model.pt'))
        model_test.load_state_dict(checkpoint['model_state_dict'])
    except:
        model_test = ResNet(input_shape=(1,3000), num_classes=11, n_feature_maps=128).to(device)
        checkpoint = torch.load(join(root, f'4_checkpoints/resnet/penguin_archive/{case}/best_model.pt'))
        model_test.load_state_dict(checkpoint['model_state_dict'])
    
    print(">", case, "<")
    for k, v in checkpoint.items():
        if "state_dict" in k:
            continue
        print(k, v)
    print()
    
    for datum in data:
        print(f">>> {case} @ {datum} <<<")

        run_test(X_path=join(root, f"3_model_input_data/resnet/penguin_archive/{datum}/X_test.csv"),
                y_path=join(root, f"3_model_input_data/resnet/penguin_archive/{datum}/Y_test.csv"),
                model_test=model_test,
                criterion=nn.CrossEntropyLoss()
                )
        print()
    


> hevc_marginally_imbalanced_800K_withB <
epoch 215
train_loss tensor(2.4897e-05, device='cuda:0', requires_grad=True)
val_loss tensor(0.6594, device='cuda:0')
train_acc 1.0
val_acc 0.8569587628865979

>>> hevc_marginally_imbalanced_800K_withB @ hevc_marginally_imbalanced_800K_withB <<<
   precision  accuracy  recall  duration
0      85.98     85.74   86.14       0.0

>>> hevc_marginally_imbalanced_800K_withB @ hevc_marginally_imbalanced_1000K_withB <<<
   precision  accuracy  recall  duration
0      85.93     85.68   86.46       0.0

>>> hevc_marginally_imbalanced_800K_withB @ hevc_marginally_imbalanced_1200K_withB <<<
   precision  accuracy  recall  duration
0      84.82     84.45   85.28       0.0

>>> hevc_marginally_imbalanced_800K_withB @ hevc_marginally_imbalanced_1500K_withB <<<
   precision  accuracy  recall  duration
0      82.87     82.54   84.11       0.0

> hevc_marginally_imbalanced_1000K_withB <
epoch 89
train_loss tensor(0.0002, device='cuda:0', requires_grad=True)
val_

In [16]:
metrics = calculate_metrics([0], [0], 0)

In [25]:
metrics['accuracy'][0]

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'type'

In [ ]:
# Load data from the DataFrame
X_test = pd.read_csv(join(root, "3_model_input_data/resnet/penguin_archive/hevc_marginally_imbalanced_800K/X_test.csv")).values
y_test = pd.read_csv(join(root, "3_model_input_data/resnet/penguin_archive/hevc_marginally_imbalanced_800K/Y_test.csv")).values

# normalize X (each example individually), 
# and expand X to include a C dimension, models expects it
X_test_mean, X_test_std = X_test.mean(axis=1, keepdims=True), X_test.std(axis=1, keepdims=True)
X_test = (X_test - X_test_mean) / X_test_std
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# and one-hot encode y, to calculate cross-entropy loss
enc = OneHotEncoder(categories='auto')
y_test = enc.fit_transform(y_test.reshape(-1, 1)).toarray()
nb_classes = len(enc.categories_[0])

# a quick stats check of our test set
print("samples:", len(X_test), ", num_classes", nb_classes)
print("labels for couple random samples: ", 
      enc.categories_[0][y_test.argmax(axis=1)][np.random.randint(0, len(y_test), 20)])

In [172]:
from sklearn.model_selection import train_test_split

X = ['News0', 'News1', 'News2', 'News3', 'News4', 'News5', 'News6', 'News7', 'News8', 'News9', 'News10', 'News11', 'News12', 'News13', 'News14', 'News15', 'News16', 'News17', 'News18', 'News19', 'News20', 'News21', 'News22', 'News23', 'News24', 'News25', 'News26', 'News27', 'News28', 'News29', 'News30', 'News31', 'News32', 'News33', 'News34', 'News35', 'News36', 'News37', 'News38', 'Sports39', 'Sports40', 'Sports41', 'Sports42', 'Sports43', 'Sports44', 'Sports45', 'Sports46', 'Sports47', 'Sports48', 'Sports49', 'Sports50', 'Sports51', 'Sports52', 'Sports53', 'Sports54', 'Sports55', 'Sports56', 'Sports57', 'Sports58', 'Sports59', 'Sports60', 'Sports61', 'Sports62', 'Sports63', 'Sports64', 'Sports65', 'Sports66', 'Sports67', 'Sports68', 'Sports69', 'Sports70', 'Sports71', 'Sports72', 'Sports73', 'Sports74', 'Sports75', 'Sports76', 'Sports77', 'Sports78', 'Sports79', 'Sports80', 'Sports81', 'Sports82', 'Sports83', 'Sports84', 'Sports85', 'Sports86', 'Sports87', 'Sports88', 'Sports89', 'Sports90', 'Sports91', 'Sports92', 'Sports93', 'Sports94', 'Sports95', 'Sports96', 'Sports97', 'Sports98', 'Sports99', 'Sports100', 'Sports101', 'Sports102', 'Sports103', 'Sports104', 'Sports105', 'Sports106', 'Sports107', 'Sports108']

(X_train, 
 X_test) = train_test_split(X, test_size=0.1, random_state=42)

In [173]:
X_train, X_test

(['Sports80',
  'News0',
  'Sports81',
  'News18',
  'Sports70',
  'Sports56',
  'Sports72',
  'Sports107',
  'Sports42',
  'News12',
  'News36',
  'Sports65',
  'News26',
  'News22',
  'News31',
  'Sports47',
  'Sports76',
  'News15',
  'Sports44',
  'Sports89',
  'Sports90',
  'News9',
  'News33',
  'Sports55',
  'Sports69',
  'News28',
  'Sports40',
  'News5',
  'Sports53',
  'Sports62',
  'Sports39',
  'News35',
  'News16',
  'Sports103',
  'News34',
  'Sports67',
  'News7',
  'Sports43',
  'Sports66',
  'Sports73',
  'News27',
  'News19',
  'Sports88',
  'Sports93',
  'News25',
  'News8',
  'Sports101',
  'Sports49',
  'News13',
  'Sports77',
  'News24',
  'News3',
  'News17',
  'News38',
  'Sports85',
  'News6',
  'Sports104',
  'Sports95',
  'Sports91',
  'Sports54',
  'Sports50',
  'Sports98',
  'Sports46',
  'Sports83',
  'Sports61',
  'Sports105',
  'Sports100',
  'Sports41',
  'Sports58',
  'Sports48',
  'Sports94',
  'Sports57',
  'Sports75',
  'News32',
  'Sports108',
  'S

samples: 1941 , num_classes 11
labels for couple random samples:  ['Knowledge' 'Sports' 'Sports' 'Entertainment' 'Music' 'Beauty'
 'Education' 'Knowledge' 'News' 'Music' 'Music' 'Cooking' 'Entertainment'
 'Movies' 'Knowledge' 'Technology' 'Cooking' 'Knowledge' 'Entertainment'
 'Education']


In [90]:
## creating data loaders
# Convert the numpy array to a PyTorch tensor
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.float32))

### Create a DataLoader from the TensorDataset
batch_size = 1024
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [91]:
# and run the test dataset through our saved model
model_test.eval()
with torch.no_grad():
    test_loss = 0     # for lr scheduler
    test_logits, test_truth = [], []
    for batch_num, (X, y) in enumerate(test_dataloader):
        print(f"{batch_num}/{len(test_dataloader)}")
        X, y = X.to(device), y.to(device)
        logits = model_test(X)
        test_logits += logits
        test_truth += y
        test_loss += criterion(logits, y) * len(y)     # properly weighting each batch loss, 
    test_loss /= len(test_dataloader.dataset)

print(test_loss)


0/2
1/2
tensor(0.6491, device='cuda:0')


In [93]:
pred = torch.stack(test_truth).max(dim=1)[1].cpu().numpy()
truth = torch.stack(test_logits).max(dim=1)[1].cpu().numpy()

print(calculate_metrics(pred, truth, 0.0))

# now calculate accuracies for each class
total = Counter(truth)      # total number of samples for each category, taken from Ground Truth
                            # so... RECALL?
correct_pred = Counter(truth[truth == pred])

stats = [ {"category": enc.categories_[0][numeric_cat],
           "correct": correct_pred[numeric_cat], 
           "total_samples": total[numeric_cat], 
           "accuracy": correct_pred[numeric_cat]/total[numeric_cat]}
            for numeric_cat in total]

for x in sorted(stats, key=lambda x: x["accuracy"], reverse=True):
    print(f"Accuracy for {x['category']:<15}= {x['accuracy']:.2%}  <--  {x['correct']}/{x['total_samples']}")



   precision  accuracy    recall  duration
0   0.849178  0.850592  0.856026       0.0
Accuracy for Education      = 94.30%  <--  149/158
Accuracy for Game           = 89.64%  <--  173/193
Accuracy for News           = 89.12%  <--  172/193
Accuracy for Knowledge      = 88.89%  <--  96/108
Accuracy for Sports         = 86.14%  <--  174/202
Accuracy for Technology     = 85.29%  <--  174/204
Accuracy for Cooking        = 84.90%  <--  163/192
Accuracy for Beauty         = 84.69%  <--  166/196
Accuracy for Entertainment  = 81.34%  <--  170/209
Accuracy for Music          = 76.81%  <--  106/138
Accuracy for Movies         = 72.97%  <--  108/148


In [35]:
from collections import Counter
total = Counter(pred)      # if total taken as pred, then calculating PRECISION
correct_pred = Counter(pred[truth == pred])

stats = [ {"category": enc.categories_[0][numeric_cat],
           "correct": correct_pred[numeric_cat], 
           "total_samples": total[numeric_cat], 
           "accuracy": correct_pred[numeric_cat]/total[numeric_cat]}
            for numeric_cat in total]

for x in sorted(stats, key=lambda x: x["accuracy"], reverse=True):
    print(f"Accuracy for {x['category']:<15}= {x['accuracy']:.2%}  <--  {x['correct']}/{x['total_samples']}")


Accuracy for Movies         = 100.00%  <--  134/134
Accuracy for Beauty         = 0.00%  <--  0/201
Accuracy for News           = 0.00%  <--  0/181
Accuracy for Technology     = 0.00%  <--  0/202
Accuracy for Cooking        = 0.00%  <--  0/203
Accuracy for Music          = 0.00%  <--  0/173
Accuracy for Game           = 0.00%  <--  0/195
Accuracy for Education      = 0.00%  <--  0/154
Accuracy for Sports         = 0.00%  <--  0/193
Accuracy for Entertainment  = 0.00%  <--  0/207
Accuracy for Knowledge      = 0.00%  <--  0/98


In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d__%H-%M-%S')


In [ ]:
import numpy as np
np.random.seed(42)
np.random.permutation(100)
